# House Price Prediction - Exploratory Data Analysis

This notebook explores the synthetic real estate dataset used to train the price prediction models.
The goal is to understand the data before building anything - look at distributions, relationships
between features, and spot any issues we need to handle.

**What we cover here:**
1. Load & inspect the raw data
2. Missing values and basic statistics
3. Price and area distributions
4. Price vs area relationship
5. Average price by district
6. Correlation matrix
7. Price by number of rooms
8. Outlier detection
9. Feature engineering preview

In [ ]:
import sys
import os

# Add the project root to Python path so we can import from src/
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

plt.rcParams['figure.figsize'] = (11, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print('Libraries loaded OK')

## 1. Load the Dataset

We generate the synthetic dataset using the helper function from `src/data_preprocessing.py`.
If the CSV already exists on disk (e.g. after running `main.py`), we load it directly.

In [ ]:
from src.data_preprocessing import generate_synthetic_data

RAW_PATH = '../data/raw/house_prices.csv'

if os.path.exists(RAW_PATH):
    df = pd.read_csv(RAW_PATH)
    print(f'Loaded existing dataset: {df.shape}')
else:
    df = generate_synthetic_data(n_samples=1500, save_path=RAW_PATH)
    print(f'Generated new dataset: {df.shape}')

## 2. Basic Inspection

In [ ]:
print(f'Shape: {df.shape[0]} rows x {df.shape[1]} columns\n')
df.head()

In [ ]:
df.info()

In [ ]:
# Summary statistics for numeric columns
df.describe()

In [ ]:
# Value counts for categorical columns
print('District breakdown:')
print(df['district'].value_counts())
print()
print('Condition breakdown:')
print(df['condition'].value_counts())

## 3. Missing Values

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0]

if missing_df.empty:
    print('No missing values found!')
else:
    print(missing_df)

We have a small percentage of missing values in `building_age`, `floor`, and `bathrooms`.
Since it's under 5%, we can safely fill them with the median (for numeric columns).
This is a simple but reasonable approach for a dataset this size.

In [ ]:
# Fill missing values
df['building_age'] = df['building_age'].fillna(df['building_age'].median())
df['floor']        = df['floor'].fillna(df['floor'].median())
df['bathrooms']    = df['bathrooms'].fillna(1)

print(f'Missing values after filling: {df.isnull().sum().sum()}')

## 4. Price Distribution

Let's start with the target variable - price. It's important to check if prices are
normally distributed or skewed. Most real estate price distributions are right-skewed
(a few very expensive properties pull the average up).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['price'], bins=45, color='steelblue', edgecolor='white')
axes[0].set_title('Price Distribution')
axes[0].set_xlabel('Price ($)')
axes[0].set_ylabel('Count')
axes[0].axvline(df['price'].median(), color='red', linestyle='--',
                linewidth=1.5, label=f'Median: ${df["price"].median():,.0f}')
axes[0].legend()

axes[1].hist(np.log1p(df['price']), bins=45, color='coral', edgecolor='white')
axes[1].set_title('Price Distribution (log scale)')
axes[1].set_xlabel('log(Price + 1)')
axes[1].set_ylabel('Count')

plt.suptitle('Distribution of House Prices', fontsize=14)
plt.tight_layout()
plt.show()

print(f'Price range:  ${df["price"].min():,.0f}  –  ${df["price"].max():,.0f}')
print(f'Mean price:   ${df["price"].mean():,.0f}')
print(f'Median price: ${df["price"].median():,.0f}')

## 5. Area Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(df['area_sqm'], bins=40, color='mediumseagreen', edgecolor='white')
ax.axvline(df['area_sqm'].median(), color='red', linestyle='--',
           linewidth=1.5, label=f'Median: {df["area_sqm"].median():.0f} sqm')
ax.set_title('Area Distribution')
ax.set_xlabel('Area (sqm)')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Area range: {df["area_sqm"].min():.0f} – {df["area_sqm"].max():.0f} sqm')

## 6. Price vs Area

This is probably the most important relationship in real estate - bigger apartments cost more.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

sc = ax.scatter(
    df['area_sqm'], df['price'],
    c=df['rooms'], cmap='viridis', alpha=0.45, s=22
)
plt.colorbar(sc, ax=ax, label='Number of Rooms')

ax.set_xlabel('Area (sqm)')
ax.set_ylabel('Price ($)')
ax.set_title('Price vs Area (coloured by number of rooms)')
plt.tight_layout()
plt.show()

# Pearson correlation
corr = df['area_sqm'].corr(df['price'])
print(f'Pearson correlation between area and price: {corr:.3f}')

There's a clear positive relationship between area and price, as expected.
Notice how larger apartments tend to have more rooms (darker dots on the right side).

## 7. Price by District

Location is often said to be the most important factor in real estate pricing.

In [ ]:
# Calculate mean and median price per district
district_stats = df.groupby('district')['price'].agg(['mean', 'median', 'count'])
district_stats.columns = ['Mean Price', 'Median Price', 'Count']
district_stats = district_stats.sort_values('Mean Price', ascending=False)
print(district_stats.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Bar chart — average price
avg = df.groupby('district')['price'].mean().sort_values(ascending=False)
avg.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white', width=0.65)
axes[0].set_title('Average Price by District')
axes[0].set_xlabel('District')
axes[0].set_ylabel('Average Price ($)')
axes[0].tick_params(axis='x', rotation=35)

# Box plot — price distribution per district
order = df.groupby('district')['price'].median().sort_values(ascending=False).index
df.boxplot(column='price', by='district', ax=axes[1], notch=False,
           boxprops=dict(color='steelblue'),
           medianprops=dict(color='red', linewidth=2))
axes[1].set_title('Price Distribution by District')
plt.suptitle('')
axes[1].set_xlabel('District')
axes[1].set_ylabel('Price ($)')
axes[1].tick_params(axis='x', rotation=35)

plt.tight_layout()
plt.show()

**Observation:** Downtown and Central are the most expensive districts, while Suburbs are the cheapest.
This makes intuitive sense - central locations have higher demand.
Location will be an important feature for the model.

## 8. Correlation Matrix

Let's check how all numeric features correlate with each other and with price.

In [ ]:
num_cols = [
    'price', 'area_sqm', 'rooms', 'bedrooms', 'bathrooms',
    'floor', 'building_age', 'distance_to_center_km',
    'has_balcony', 'has_parking',
]

corr = df[num_cols].corr()

# Show only correlations with price, sorted
price_corr = corr['price'].drop('price').sort_values(ascending=False)
print('Correlations with price:')
print(price_corr.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='coolwarm',
    mask=mask, square=True, linewidths=0.5, ax=ax,
    annot_kws={'size': 9}
)
ax.set_title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

**Observations:**
- `area_sqm` has the strongest correlation with price - expected, bigger = more expensive
- `rooms` and `bedrooms` are highly correlated with each other (multicollinearity)
- `distance_to_center_km` has a **negative** correlation with price- further from center = cheaper
- `building_age` has a **negative** correlation - older buildings tend to be cheaper

## 9. Price by Number of Rooms

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Boxplot
df.boxplot(column='price', by='rooms', ax=axes[0],
           boxprops=dict(color='steelblue'),
           medianprops=dict(color='red', linewidth=2))
axes[0].set_title('Price by Rooms (boxplot)')
plt.suptitle('')
axes[0].set_xlabel('Number of Rooms')
axes[0].set_ylabel('Price ($)')

# Average price per room count
avg_by_rooms = df.groupby('rooms')['price'].mean()
avg_by_rooms.plot(kind='bar', ax=axes[1], color='mediumseagreen',
                  edgecolor='white', width=0.65)
axes[1].set_title('Average Price by Rooms')
axes[1].set_xlabel('Number of Rooms')
axes[1].set_ylabel('Average Price ($)')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

print('Average price per room count:')
print(avg_by_rooms.apply(lambda x: f'${x:,.0f}'))

## 10. Outlier Analysis

Outliers in price and area can hurt regression models. Let's identify them.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].boxplot(df['price'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='lightblue'),
                medianprops=dict(color='red', linewidth=2))
axes[0].set_title('Price — Outlier Check')
axes[0].set_ylabel('Price ($)')

axes[1].boxplot(df['area_sqm'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='lightgreen'),
                medianprops=dict(color='red', linewidth=2))
axes[1].set_title('Area — Outlier Check')
axes[1].set_ylabel('Area (sqm)')

plt.tight_layout()
plt.show()

In [ ]:
# IQR-based outlier count
for col in ['price', 'area_sqm']:
    Q1  = df[col].quantile(0.25)
    Q3  = df[col].quantile(0.75)
    IQR = Q3 - Q1
    low  = Q1 - 1.5 * IQR
    high = Q3 + 1.5 * IQR
    n_outliers = ((df[col] < low) | (df[col] > high)).sum()
    print(f'{col}: {n_outliers} potential outliers  '
          f'(IQR fence: [{low:,.0f}, {high:,.0f}])')

We use the 1st–99th percentile clip in `data_preprocessing.py` to remove the most extreme values
without discarding too much data. IQR would be more aggressive.

## 11. Feature Engineering Preview

A quick look at the new features we add before modelling.

In [ ]:
from src.data_preprocessing import engineer_features

df_fe = engineer_features(df)

new_cols = ['area_per_room', 'floor_ratio', 'building_age_category',
            'district_encoded', 'condition_encoded', 'age_category_encoded', 'price_per_sqm']

print('New columns added:')
df_fe[new_cols].head(8)

In [ ]:
# Price per sqm by district (for insight — not used in training)
fig, ax = plt.subplots(figsize=(10, 5))

ppsqm = df_fe.groupby('district')['price_per_sqm'].mean().sort_values(ascending=False)
ppsqm.plot(kind='bar', ax=ax, color='darkorange', edgecolor='white', width=0.65)
ax.set_title('Average Price per sqm by District')
ax.set_xlabel('District')
ax.set_ylabel('Average Price per sqm ($)')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## Summary

Key takeaways from the EDA:

| Finding | Impact |
|---|---|
| Price is right-skewed | Tree models handle this better than linear regression |
| Strong area–price correlation | `area_sqm` will be the most important feature |
| District matters a lot | Need proper encoding for the model |
| Distance from centre is negative | Confirms the location effect |
| Building age is negative | Older = cheaper (on average) |
| Rooms and bedrooms are correlated | Could cause multicollinearity in linear models |

The data looks clean enough to train on. Let's move to model training - run `main.py` from the project root.